# DermaAI — ConvNeXt-Tiny training (Google Colab, T4 / L4)

**Purpose:** train a ConvNeXt-Tiny binary classifier on HAM10000 that is a drop-in
replacement candidate for the current production MobileNetV2 model.

| Item | Value |
|---|---|
| Architecture | `ConvNeXt-Tiny` (ImageNet) + GAP + Dropout + `Dense(1, sigmoid)` |
| Input | `224x224x3`, float32 in `[0, 255]` (rescaling is **inside** the model) |
| Label `1` = Cancer | `dx` in `mel`, `bcc`, `akiec` |
| Label `0` = Non-Cancer | `dx` in `nv`, `bkl`, `vasc`, `df` |
| Split | lesion-level stratified **70 / 15 / 15**, `random_state=42` |
| Metrics | accuracy, precision, recall (cancer sensitivity), F1, ROC-AUC, confusion matrix |
| Threshold | chosen on **validation only**, then test evaluated **once** |

### Rules this notebook obeys
* Labels come **only** from `HAM10000_metadata.csv` (`dx` column). Filenames are **never** ground truth.
* The threshold is selected on the **validation** split. The **test** split is scored exactly **once**,
  at that single frozen threshold.
* The existing production model `model/skin_cancer_mobilenetv2.h5` is **never** touched.
* Nothing is auto-deployed. The notebook only *prints* the handover steps.

### How to run
1. Colab -> **Runtime -> Change runtime type -> GPU (T4 GPU or L4 GPU)**.
2. Runtime -> **Run all**.
3. Everything checkpoints to `MyDrive/DermaAI_convnext/` every epoch. If Colab disconnects,
   re-run the notebook: it auto-resumes from the newest checkpoint on Drive.

In [ ]:
# ---------------------------------------------------------------- dependencies
import subprocess, sys

def _tf_tuple():
    try:
        import tensorflow as tf
        return tuple(int(p) for p in tf.__version__.split(".")[:2])
    except Exception:
        return (0, 0)

# ConvNeXtTiny needs TF >= 2.12; upgrade only if the runtime ships something older.
if _tf_tuple() < (2, 16):
    print("Upgrading TensorFlow -> 2.20.0 (ConvNeXtTiny unavailable in the runtime version)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                    "tensorflow==2.20.0"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "scikit-learn", "pandas", "pillow", "kagglehub", "matplotlib", "seaborn"],
               check=False)

import os, glob, json, math, random
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             classification_report)

Image.MAX_IMAGE_PIXELS = None
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE
print("TF", tf.__version__, "| sklearn", sklearn.__version__, "| pandas", pd.__version__)

In [ ]:
# ---------------------------------------------------------------- GPU check
GPUS = tf.config.list_physical_devices("GPU")
print("=" * 66)
print("GPU CHECK")
print("=" * 66)
if not GPUS:
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime -> Change runtime type -> GPU (T4 or L4), "
        "then re-run this cell.")

for g in GPUS:
    print("  device :", g.name)
    try:
        print("  details:", tf.config.experimental.get_device_details(g))
    except Exception as exc:
        print("  details unavailable:", exc)
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True)
print("  nvidia-smi:", (smi.stdout or smi.stderr).strip())

GPU_NAME = GPUS[0].name.upper()
print("\nMixed precision (float16 compute, float32 storage):")
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("  global policy:", tf.keras.mixed_precision.global_policy())

---
### 1. Google Drive (checkpointing + resume)

Checkpoints, logs, the best model and all metrics JSON go to
`MyDrive/DermaAI_convnext/`. Drive I/O is slower than local disk, but it is what makes a
Colab disconnect survivable.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/DermaAI_convnext"
CKPT_DIR  = os.path.join(DRIVE_DIR, "ckpt")
LOG_DIR   = os.path.join(DRIVE_DIR, "logs")
BEST_PATH = os.path.join(DRIVE_DIR, "convnext_tiny_best.keras")
BEST_H5   = os.path.join(DRIVE_DIR, "convnext_tiny_best.h5")
PROGRESS  = os.path.join(DRIVE_DIR, "progress.json")

for d in (DRIVE_DIR, CKPT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print("DRIVE_DIR :", DRIVE_DIR)
print("existing checkpoints:", len(glob.glob(os.path.join(CKPT_DIR, "epoch_*.keras"))))
if os.path.exists(PROGRESS):
    print("progress.json:", open(PROGRESS).read())

In [ ]:
# ---------------------------------------------------------------- config
INPUT_SIZE = (224, 224)
BATCH_SIZE = 32

CANCER_CLASSES = ["mel", "bcc", "akiec"]          # dx codes -> label 1
CLASS_NAMES    = ["Non-Cancer", "Cancer"]

STAGE1_EPOCHS = 10      # head only, backbone frozen
STAGE2_EPOCHS = 24      # fine-tune last N stages (+ full model optional)
UNFREEZE_STAGES = 2     # how many trailing ConvNeXt stages to unfreeze

LR_HEAD      = 1e-3
LR_BACKBONE  = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.02

KEEP_LAST_CKPTS = 5      # prune older per-epoch checkpoints (each full model ~114 MB)
AUTOREPLACE_PRODUCTION = False   # hard guard: never overwrite mobilenetv2 from here

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(json.dumps({k: v for k, v in locals().items()
                  if k.isupper() and isinstance(v, (int, float, str, bool, list, tuple))},
                 indent=2, default=str))

---
### 2. HAM10000 dataset

Sources are tried in order and the first one that yields a usable tree wins:

1. a local / Drive copy of HAM10000 (fastest, survives restarts),
2. `kagglehub: eliocordeiropereira/skin-cancer-the-ham10000-dataset`
   — contains `HAM10000_metadata.csv` **and** `HAM10000_images_part_1/2`,
   which is the mirror the current production model was trained from,
3. `kagglehub: kmader/skin-cancer-mnist-ham10000` — class folders only.

In [ ]:
# ---------------------------------------------------------------- dataset
KAGGLE_HANDLES = [
    "eliocordeiropereira/skin-cancer-the-ham10000-dataset",
    "kmader/skin-cancer-mnist-ham10000",
]
LOCAL_CANDIDATES = [
    "/content/HAM10000",
    "/content/ham10000",
    "/content/HAM10000_dataset",
    "/content/drive/MyDrive/HAM10000",
    "/content/drive/MyDrive/ham10000",
    "/content/drive/MyDrive/DermaAI/HAM10000",
    "/content/drive/MyDrive/DermaAI_convnext/HAM10000",
]


def _jpg_count(root):
    return len(glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True))


def resolve_dataset_root():
    for c in LOCAL_CANDIDATES:
        if os.path.isdir(c) and _jpg_count(c) > 100:
            print("  using local copy:", c)
            return c
    import kagglehub
    errs = []
    for h in KAGGLE_HANDLES:
        try:
            p = kagglehub.dataset_download(h)
            print("  downloaded", h, "->", p)
            if _jpg_count(p) > 100:
                return p
        except Exception as exc:
            print("  failed", h, ":", exc)
            errs.append(h)
    raise RuntimeError("Could not obtain HAM10000. Tried local paths + " + ", ".join(errs))


DATASET_ROOT = resolve_dataset_root()

# --- metadata CSV (authoritative labels: dx + lesion_id). Never use filenames as labels.
csv_hits = [p for p in glob.glob(os.path.join(DATASET_ROOT, "**", "*.csv"), recursive=True)
            if "metadata" in os.path.basename(p).lower()]
METADATA_CSV = csv_hits[0] if csv_hits else None
print("\nDATASET_ROOT  :", DATASET_ROOT)
print("METADATA_CSV  :", METADATA_CSV or "*** NOT FOUND — lesion-level grouping unavailable ***")
print("jpg files     :", _jpg_count(DATASET_ROOT))

In [ ]:
# ---------------------------------------------------------------- 70/15/15 split
def _build_index(root):
    '''image_id -> absolute path (recursive, robust to nested folders).'''
    idx = {}
    for p in glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True):
        idx[os.path.splitext(os.path.basename(p))[0]] = p
    return idx


def build_dataframes():
    index = _build_index(DATASET_ROOT)

    if METADATA_CSV:
        meta = pd.read_csv(METADATA_CSV)
        meta["image_id"] = meta["image_id"].astype(str).str.strip()
        meta["path"] = meta["image_id"].map(index)
        meta = meta[meta["path"].notna()].reset_index(drop=True)
        if len(meta) < 5000:
            print("  metadata join matched only %d images -> ignoring metadata CSV" % len(meta))
        else:
            meta["label"] = meta["dx"].apply(lambda x: 1 if x in CANCER_CLASSES else 0)
            meta["group"] = meta["lesion_id"].astype(str)   # lesion-level grouping key
            meta["split_by"] = "lesion_id (metadata CSV)"
            return meta

    # Fallback: class folders only -> image-level stratified split (NOT lesion-disjoint).
    rows = []
    for dx in sorted(set(CANCER_CLASSES) | {"nv", "bkl", "vasc", "df"}):
        for p in glob.glob(os.path.join(DATASET_ROOT, "**", dx, "*.jpg"), recursive=True):
            rows.append({"image_id": os.path.splitext(os.path.basename(p))[0],
                         "path": p, "dx": dx,
                         "lesion_id": os.path.splitext(os.path.basename(p))[0]})
    if not rows:
        stems_in_dir = {}
        for d in os.listdir(DATASET_ROOT):
            full = os.path.join(DATASET_ROOT, d)
            if os.path.isdir(full):
                for j in glob.glob(os.path.join(full, "*.jpg")):
                    stems_in_dir[os.path.splitext(os.path.basename(j))[0]] = d
        for p in glob.glob(os.path.join(DATASET_ROOT, "**", "*.jpg"), recursive=True):
            stem = os.path.splitext(os.path.basename(p))[0]
            dx = stems_in_dir.get(stem)
            if dx:
                rows.append({"image_id": stem, "path": p, "dx": dx, "lesion_id": stem})
    meta = pd.DataFrame(rows).drop_duplicates("path").reset_index(drop=True)
    meta["label"] = meta["dx"].apply(lambda x: 1 if x in CANCER_CLASSES else 0)
    meta["group"] = meta["lesion_id"].astype(str)
    meta["split_by"] = "image_id (FALLBACK - no metadata CSV, lesions may cross splits)"
    return meta


meta = build_dataframes()
meta["label_name"] = meta["label"].map({0: "Non-Cancer", 1: "Cancer"})
print("grouping used :", meta["split_by"].iloc[0])
print("total images  : %d | unique groups: %d" % (len(meta), meta["group"].nunique()))
print("dx counts     :", meta["dx"].value_counts().to_dict())

# lesion-level stratified 70/15/15, seed 42 (identical logic to skin_cancer1.ipynb)
grp = meta.groupby("group")["label"].max().reset_index()
g_train, g_temp = train_test_split(grp, test_size=0.30, stratify=grp["label"], random_state=SEED)
g_val,   g_test  = train_test_split(g_temp, test_size=0.50, stratify=g_temp["label"], random_state=SEED)

df_train = meta[meta["group"].isin(set(g_train["group"]))].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df_val   = meta[meta["group"].isin(set(g_val["group"]))].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
df_test  = meta[meta["group"].isin(set(g_test["group"]))].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

REF = {  # numbers produced by the verified production split
    "train": (7010, 5229, {"Non-Cancer": 5641, "Cancer": 1369}),
    "val":   (1500, 1120, {"Non-Cancer": 1203, "Cancer": 297}),
    "test":  (1505, 1121, {"Non-Cancer": 1217, "Cancer": 288}),
}
print("\n%-6s %8s %8s %8s  %s" % ("split", "images", "groups", "overlap", "class counts"))
overlap_ok = True
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    ov = (len(set(df_train["group"]) & set(df_val["group"])) +
          len(set(df_train["group"]) & set(df_test["group"])) +
          len(set(df_val["group"]) & set(df_test["group"])))
    overlap_ok &= (ov == 0)
    print("%-6s %8d %8d %8d  %s" % (name, len(df), df["group"].nunique(), ov,
                                   df["label_name"].value_counts().to_dict()))
    r = REF[name]
    print("       reference      %8d %8d %8s  %s" % (r[0], r[1], "-", r[2]))
print("\nSPLIT GROUP DISJOINT :", overlap_ok)
print("MATCHES PRODUCTION SPLIT:", all(
    len(d) == REF[n][0] and d["group"].nunique() == REF[n][1]
    and d["label_name"].value_counts().to_dict() == REF[n][2]
    for n, d in (("train", df_train), ("val", df_val), ("test", df_test))))
if not overlap_ok:
    raise RuntimeError("Group leakage between splits — aborting.")

---
### 3. Input pipeline, preprocessing and class balancing

**Preprocessing contract (must match the served model exactly):**
`RGB -> resize 224x224 -> float32 in [0, 255] -> ConvNeXt internal Rescaling(1/127.5, -1.0) -> [-1, 1]`

The rescaling lives *inside* the model, so a saved ConvNeXt file can be served without
re-implementing preprocessing. It does **not** match the current MobileNetV2 contract
(`/255` in `app.py`); the handover cell at the end spells out that one-line change.

**Class balancing:** the training split is ~80 / 20. `tf.data.Dataset.sample_from_datasets`
with equal weights resamples it to a 50 / 50 stream every epoch, so the minority cancer
class is oversampled with augmentation rather than down-sampled.

In [ ]:
# ---------------------------------------------------------------- tf.data pipeline
tf.keras.utils.set_random_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE
N_TRAIN  = len(df_train)
STEPS_PER_EPOCH = math.ceil(N_TRAIN / BATCH_SIZE)   # one pass over the 70% split


def _decode(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)          # (H, W, 3) uint8
    img = tf.image.resize(img, INPUT_SIZE, method="bilinear")
    img = tf.image.convert_image_dtype(img, tf.float32)  # -> [0, 1]
    img = tf.clip_by_value(img * 255.0, 0.0, 255.0)     # -> [0, 255] raw RGB float
    return img, tf.cast(label, tf.float32)


_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomFlip("vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name="augment")


def _decode_aug(path, label):
    img, lab = _decode(path, label)
    img = _augment(tf.expand_dims(img, 0), training=True)[0]
    return tf.clip_by_value(img, 0.0, 255.0), lab


def make_ds(df, training):
    '''Order-preserving for eval (labels are aligned positionally), shuffled for train.'''
    paths  = df["path"].to_numpy().astype(str)
    labels = df["label"].to_numpy().astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_decode_aug if training else _decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.repeat().shuffle(N_TRAIN, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


def make_balanced_ds(df):
    '''50/50 cancer / non-cancer resampling of the TRAIN split (training augmentation on).'''
    pos = df[df["label"] == 1]
    neg = df[df["label"] == 0]
    print("  balancing: %d cancer -> oversampled to match %d non-cancer" % (len(pos), len(neg)))

    def one(sub):
        return (tf.data.Dataset.from_tensor_slices(
                    (sub["path"].to_numpy().astype(str), sub["label"].to_numpy().astype(np.float32)))
                .map(_decode_aug, num_parallel_calls=AUTOTUNE).shuffle(
                    len(sub) * 4, seed=SEED, reshuffle_each_iteration=True))

    ds = tf.data.Dataset.sample_from_datasets(
        [one(pos), one(neg)], weights=[0.5, 0.5], seed=SEED, stop_on_empty_dataset=True)
    return ds.shuffle(N_TRAIN * 2, seed=SEED, reshuffle_each_iteration=True) \
            .batch(BATCH_SIZE).prefetch(AUTOTUNE)


print("building datasets …")
train_bal = make_balanced_ds(df_train)
val_ds    = make_ds(df_val,   training=False)
test_ds   = make_ds(df_test,  training=False)

x_chk, y_chk = next(iter(val_ds))
print("\ntrain (balanced) element_spec:", train_bal.element_spec)
print("val   element_spec           :", val_ds.element_spec)
print("val batch shape/dtype        :", x_chk.shape, x_chk.dtype,
      "| range [%.1f, %.1f]" % (float(tf.reduce_min(x_chk)), float(tf.reduce_max(x_chk))))
print("y unique                     :", np.unique(y_chk.numpy()))
print("STEPS_PER_EPOCH              :", STEPS_PER_EPOCH)

---
### 4. Model

`ConvNeXtTiny(include_top=False, weights="imagenet")` + `Dropout` + `Dense(1, sigmoid)`.

Keras has moved ConvNeXt's internal `Rescaling(1/127.5, -1.0)` and `Normalization`
layers between releases, so the cell below **probes the installed TF/Keras** and only
adds a `Rescaling` layer when it is actually missing. It also skips a second
`GlobalAveragePooling2D` when ConvNeXtTiny already ends with one.

In [ ]:
# ---------------------------------------------------------------- build model
from tf.keras.applications.convnext import ConvNeXtTiny


def build_model():
    inp = tf.keras.Input(shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3), name="input_image")
    x = inp
    base = ConvNeXtTiny(include_top=False, weights="imagenet",
                        include_preprocessing=False,
                        input_shape=(INPUT_SIZE[0], INPUT_SIZE[1], 3))
    has_rescale = any(isinstance(l, tf.keras.layers.Rescaling) for l in base.layers)
    if not has_rescale:
        x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="rescale")(x)
    print("ConvNeXt internal Rescaling present:", has_rescale, "-> local Rescaling added:", not has_rescale)

    x = base(x)
    print("ConvNeXt tail layer:", type(base.layers[-1]).__name__)
    x = tf.keras.layers.Dropout(0.30, name="dropout")(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32", name="cancer")(x)
    return tf.keras.Model(inp, out, name="dermai_convnext_tiny")


model = build_model()
model.summary()
print("input :", model.input_shape, model.input_dtype)
print("output:", model.output_shape, model.output_dtype)

In [ ]:
# ---------------------------------------------------------------- callbacks + resume
def make_callbacks(tag, monitor="val_auc", patience=8, use_plateau=True):
    class PruneCkpts(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            fs = sorted(glob.glob(os.path.join(CKPT_DIR, "epoch_*.keras")))
            for stale in fs[:-KEEP_LAST_CKPTS]:
                try:
                    os.remove(stale)
                except OSError:
                    pass

    def save_progress(stage):
        def _cb(epoch, logs=None):
            tmp = PROGRESS + ".tmp"
            with open(tmp, "w") as fh:
                json.dump({"stage": stage, "last_epoch": int(epoch),
                           "monitor": monitor,
                           "best": (float(logs[monitor]) if logs and monitor in logs else None),
                           "updated": str(pd.Timestamp.utcnow())}, fh)
            os.replace(tmp, PROGRESS)
        return _cb

    cbs = [
        # checkpoint EVERY epoch (full model, so a disconnect resumes exactly)
        tf.keras.callbacks.ModelCheckpoint(os.path.join(CKPT_DIR, "epoch_{epoch:03d}.keras"),
                                            save_weights_only=False, save_best_only=False,
                                            verbose=1),
        tf.keras.callbacks.ModelCheckpoint(BEST_PATH, monitor=monitor, mode="max",
                                            save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor=monitor, mode="max", patience=patience,
                                         restore_best_weights=True, verbose=1),
        tf.keras.callbacks.CSVLogger(os.path.join(LOG_DIR, "history_%s.csv" % tag)),
        save_progress(tag),
        PruneCkpts(),
    ]
    if use_plateau:
        # only valid with a scalar learning rate (stage 1). Stage 2 uses per-variable
        # cosine schedules that Keras steps itself.
        cbs.insert(3, tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, mode="max", factor=0.3, patience=3, min_lr=1e-6, verbose=1))
    return cbs


def latest_checkpoint():
    fs = sorted(glob.glob(os.path.join(CKPT_DIR, "epoch_*.keras")))
    return fs[-1] if fs else None


progress_state = {}
if os.path.exists(PROGRESS):
    try:
        progress_state = json.load(open(PROGRESS))
    except Exception:
        progress_state = {}

resume_ckpt = latest_checkpoint()
if resume_ckpt and progress_state:
    print("RESUMING from", os.path.basename(resume_ckpt), "| state:", progress_state)
    model = tf.keras.models.load_model(resume_ckpt, compile=False)
else:
    print("Fresh start (no usable checkpoint on Drive).")

---
### 5. Stage 1 — train the head, backbone frozen

Backbone frozen, `LR = 1e-3` AdamW, 10 epochs. This gets the classifier head into a sane
region before any backbone weight moves.

In [ ]:
# ---------------------------------------------------------------- STAGE 1
_resumed_stage = progress_state.get("stage")
_last_epoch = int(progress_state.get("last_epoch", -1)) if progress_state else -1

if _resumed_stage == "stage2":
    stage1_epochs, STAGE1_DONE = 0, True
    print("STAGE 1 skipped: Drive says stage 2 already reached.")
elif _resumed_stage == "stage1" and _last_epoch >= 0:
    stage1_epochs = max(STAGE1_EPOCHS - (_last_epoch + 1), 0)
    STAGE1_DONE = _last_epoch + 1 >= STAGE1_EPOCHS
    print("STAGE 1 resuming: %d/%d epochs already done -> %d left"
          % (_last_epoch + 1, STAGE1_EPOCHS, stage1_epochs))
else:
    stage1_epochs, STAGE1_DONE = STAGE1_EPOCHS, False
    print("STAGE 1 fresh: %d epochs" % stage1_epochs)

if stage1_epochs > 0:
    for layer in model.layers:
        if layer.name in ("input_image", "rescale"):
            layer.trainable = True
        elif isinstance(layer, tf.keras.models.Model):
            layer.trainable = False
        else:
            layer.trainable = True

    n_trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
    n_total     = int(model.count_params())
    print("STAGE 1 | trainable params %d / %d (%.2f%%)" % (n_trainable, n_total, 100 * n_trainable / n_total))

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=LR_HEAD, weight_decay=WEIGHT_DECAY),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc", curve="ROC")],
    )
    history_s1 = model.fit(
        train_bal, steps_per_epoch=STEPS_PER_EPOCH, epochs=stage1_epochs,
        validation_data=val_ds,
        callbacks=make_callbacks("stage1"), verbose=2,
    )
    STAGE1_DONE = True
    print("STAGE 1 finished. best val_auc = %.4f" % max(history_s1.history["val_auc"]))

---
### 6. Stage 2 — fine-tune

The trailing `UNFREEZE_STAGES` ConvNeXt stages are unfrozen (or the whole backbone with
`UNFREEZE_STAGES = 4`) and trained with a discriminative learning rate:
head `1e-3` vs backbone `1e-5`, cosine decay to 5% of each base LR, 24 epochs,
`label_smoothing = 0.02`.

The two cosine schedules are attached as *per-variable* learning rates on `AdamW`
(head `1e-3`, backbone `1e-5`), which Keras steps itself on every batch. Because of
that, stage 2 deliberately has **no** `ReduceLROnPlateau` / `LearningRateScheduler`
callback — either one would overwrite the per-variable schedule with a scalar.

Stage 2 is also the only stage that watches `val_auc`, and `EarlyStopping(patience=6)`
with `restore_best_weights=True` means the last thing standing in the model at the end of
the cell is the best-validation-AUC weights seen.

In [ ]:
# ---------------------------------------------------------------- STAGE 2
STAGE2_STAGES = ["stem", "stage1", "stage2", "stage3", "stage4"]
to_unfreeze = STAGE2_STAGES[-UNFREEZE_STAGES:] if 0 < UNFREEZE_STAGES <= 5 else STAGE2_STAGES

if not STAGE1_DONE and progress_state.get("stage") != "stage2":
    raise RuntimeError("Stage 1 has not run in this session and Drive has no stage-2 state.")

base_layer = next(l for l in model.layers if isinstance(l, tf.keras.models.Model))
BACKBONE_PREFIX = base_layer.name + "/"
print("unfreezing ConvNeXt stages:", to_unfreeze)
for layer in base_layer.layers:
    layer.trainable = any(layer.name.startswith(s) for s in to_unfreeze)
frozen = [l.name for l in base_layer.layers if not l.trainable]
print("still frozen:", frozen)

base_trainable = sum(int(np.prod(w.shape)) for l in base_layer.layers
                     for w in l.trainable_weights)
base_total = int(sum(np.prod(w.shape) for l in base_layer.layers for w in l.weights))
print("backbone trainable %d / %d (%.2f%%)" % (base_trainable, base_total,
                                               100 * base_trainable / max(base_total, 1)))

stages_done = _last_epoch if _resumed_stage == "stage2" else -1
remaining = max(STAGE2_EPOCHS - (stages_done + 1), 0)
print("stage 2 epochs remaining:", remaining)

if remaining > 0:
    total_steps = max(remaining * STEPS_PER_EPOCH, 1)
    # Per-variable cosine schedules: head decays from LR_HEAD, backbone from LR_BACKBONE.
    # Keras steps a list-valued learning rate itself, so no LR callbacks are used here
    # (ReduceLROnPlateau / LearningRateScheduler would overwrite the list).
    lrs = []
    for w in model.trainable_weights:
        base_lr = LR_BACKBONE if w.name.startswith(BACKBONE_PREFIX) else LR_HEAD
        lrs.append(tf.keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=base_lr, decay_steps=total_steps, alpha=0.05))
    print("per-variable LR heads: head=%g backbone=%g over %d steps"
          % (LR_HEAD, LR_BACKBONE, total_steps))

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lrs, weight_decay=WEIGHT_DECAY,
                                             name="adamw_discriminative"),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTH),
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc", curve="ROC")],
    )
    history_s2 = model.fit(
        train_bal, steps_per_epoch=STEPS_PER_EPOCH, epochs=remaining,
        validation_data=val_ds,
        callbacks=make_callbacks("stage2", patience=6, use_plateau=False), verbose=2,
    )
    print("STAGE 2 finished. best val_auc = %.4f" % max(history_s2.history["val_auc"]))
else:
    print("STAGE 2 already complete.")

print("\ncheckpoints on Drive:", sorted(os.path.basename(p) for p in glob.glob(os.path.join(CKPT_DIR, "epoch_*.keras"))))
print("best model          :", BEST_PATH, "exists =", os.path.exists(BEST_PATH))

In [ ]:
# ---------------------------------------------------------------- load best model
if not os.path.exists(BEST_PATH):
    raise RuntimeError("No best model was saved — fit() did not complete successfully.")

best_model = tf.keras.models.load_model(BEST_PATH, compile=False)
best_model.summary(line_length=110)
print("\nbest model input :", best_model.input_shape, best_model.input_dtype)
print("best model output:", best_model.output_shape, best_model.output_dtype)
assert tuple(best_model.input_shape[1:3]) == (INPUT_SIZE[0], INPUT_SIZE[1]), "wrong input size"
assert tuple(best_model.output_shape[1:]) == (1,), "wrong output size"
print("ASSERTIONS OK — model loaded and shapes are correct.")

In [ ]:
# ---------------------------------------------------------------- predict (VAL + TEST)
def predict_probs(model, ds, df, batch=64):
    '''ds must be order-preserving (no shuffle) so predictions line up with df's labels.'''
    y_true = df["label"].to_numpy().astype(np.int32)
    p = model.predict(ds, verbose=1).reshape(-1).astype(np.float64)
    assert p.shape == y_true.shape, (p.shape, y_true.shape)
    return p, y_true


y_val_proba, y_val = predict_probs(best_model, val_ds, df_val)
y_test_proba, y_test = predict_probs(best_model, test_ds, df_test)

df_val_probs  = df_val.assign(proba=y_val_proba)
df_test_probs = df_test.assign(proba=y_test_proba)
print("\nVAL  n=%d  mean_p(cancer)=%.4f  mean_p(non-cancer)=%.4f" % (
    len(y_val), y_val_proba[y_val == 1].mean(), y_val_proba[y_val == 0].mean()))
print("TEST n=%d  mean_p(cancer)=%.4f  mean_p(non-cancer)=%.4f" % (
    len(y_test), y_test_proba[y_test == 1].mean(), y_test_proba[y_test == 0].mean()))

np.save(os.path.join(DRIVE_DIR, "val_proba.npy"), y_val_proba)
np.save(os.path.join(DRIVE_DIR, "test_proba.npy"), y_test_proba)

---
### 7. Threshold selection — **validation split only**

The grid is swept on `y_val` / `y_val_proba` and nothing else. The chosen `THRESHOLD`
is frozen here and applied once to the test split in the next cell.

Selection rule: `argmax F1` on validation (same rule used for the current production
model). The accuracy-optimal threshold and a clinically conservative
`sensitivity >= 0.95` threshold are printed for review but are **not** used for testing.

In [ ]:
# ---------------------------------------------------------------- threshold (VAL ONLY)
GRID = np.arange(0.05, 0.955, 0.005)


def sweep(y, p, grid=GRID):
    rows = []
    for t in grid:
        pred = (p >= t).astype(int)
        rows.append({"threshold": float(t),
                     "accuracy": accuracy_score(y, pred),
                     "precision": precision_score(y, pred, zero_division=0),
                     "recall": recall_score(y, pred, zero_division=0),
                     "f1": f1_score(y, pred, zero_division=0)})
    return pd.DataFrame(rows)


val_sweep = sweep(y_val, y_val_proba)
i_f1  = int(val_sweep["f1"].idxmax())
i_acc = int(val_sweep["accuracy"].idxmax())
i_sens = val_sweep[val_sweep["recall"] >= 0.95]
i_s95 = int(i_sens["f1"].idxmax()) if len(i_sens) else i_f1

THRESHOLD = float(val_sweep.loc[i_f1, "threshold"])
THRESHOLD_ACC_OPT = float(val_sweep.loc[i_acc, "threshold"])
THRESHOLD_SENS95   = float(val_sweep.loc[i_s95, "threshold"])

print("VAL sweep, argmax F1           : t=%.3f  acc=%.4f  prec=%.4f  rec=%.4f  f1=%.4f"
      % (THRESHOLD, val_sweep.loc[i_f1, "accuracy"], val_sweep.loc[i_f1, "precision"],
         val_sweep.loc[i_f1, "recall"], val_sweep.loc[i_f1, "f1"]))
print("VAL sweep, argmax accuracy     : t=%.3f  f1=%.4f" % (THRESHOLD_ACC_OPT, val_sweep.loc[i_acc, "f1"]))
print("VAL sweep, sensitivity >= 0.95 : t=%.3f  prec=%.4f  f1=%.4f"
      % (THRESHOLD_SENS95, val_sweep.loc[i_s95, "precision"], val_sweep.loc[i_s95, "f1"]))
print("\nFROZEN THRESHOLD (from validation only):", THRESHOLD)
print("NOTE: app.py keeps its own THRESHOLD = 0.5. This notebook does not change app.py.")

val_sweep.to_csv(os.path.join(LOG_DIR, "val_threshold_sweep.csv"), index=False)

In [ ]:
# ---------------------------------------------------------------- metric helpers
def metrics_at(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(thr), "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "cancer_sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "f1": float(f1_score(y, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, p)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def print_metrics(tag, m):
    print("\n" + "=" * 66)
    print("%s  (threshold = %.4f)" % (tag, m["threshold"]))
    print("=" * 66)
    print("  accuracy          : %.4f" % m["accuracy"])
    print("  precision         : %.4f" % m["precision"])
    print("  recall            : %.4f" % m["recall"])
    print("  cancer sensitivity: %.4f" % m["cancer_sensitivity"])
    print("  specificity       : %.4f" % m["specificity"])
    print("  F1                : %.4f" % m["f1"])
    print("  ROC-AUC           : %.4f" % m["roc_auc"])
    print("  confusion matrix  : tn=%d fp=%d fn=%d tp=%d  (n=%d)" % (m["tn"], m["fp"], m["fn"], m["tp"], m["n"]))
    print(classification_report(y_val if tag.startswith("VALIDATION") else y_test,
                                (y_val_proba if tag.startswith("VALIDATION") else y_test_proba) >= m["threshold"],
                                target_names=CLASS_NAMES, digits=4, zero_division=0))


val_metrics  = metrics_at(y_val,  y_val_proba,  THRESHOLD)
print_metrics("VALIDATION (used to pick the threshold)", val_metrics)

---
### 8. Test evaluation — **run once**, at the frozen validation threshold

Nothing below re-tunes anything on test.

In [ ]:
# ---------------------------------------------------------------- TEST (single evaluation)
test_metrics = metrics_at(y_test, y_test_proba, THRESHOLD)
print_metrics("TEST (single pass, threshold frozen from validation)", test_metrics)

# per-dx breakdown — dx comes from metadata, not from filenames
def per_dx(df, thr):
    pred = (df["proba"].to_numpy() >= thr).astype(int)
    t = df.assign(pred=pred).groupby("dx")["pred"].agg(["sum", "count"])
    t["detected_rate"] = t["sum"] / t["count"]
    t["binary_class"]  = t.index.map(lambda d: "Cancer" if d in CANCER_CLASSES else "Non-Cancer")
    return t.sort_values("binary_class")


print("\nper-dx detection rate @ threshold %.3f (TEST):" % THRESHOLD)
print(per_dx(df_test_probs, THRESHOLD).to_string())

mel = df_test_probs[df_test_probs["dx"] == "mel"]
MEL_SENSITIVITY = float((mel["proba"].to_numpy() >= THRESHOLD).mean()) if len(mel) else float("nan")
print("\nmelanoma-only sensitivity (dx=mel, n=%d): %.4f" % (len(mel), MEL_SENSITIVITY))

In [ ]:
# ---------------------------------------------------------------- plots + save figures
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))

fpr_v, tpr_v, _ = roc_curve(y_val, y_val_proba)
fpr_t, tpr_t, _ = roc_curve(y_test, y_test_proba)
ax[0].plot(fpr_v, tpr_v, label="val  AUC=%.4f" % val_metrics["roc_auc"])
ax[0].plot(fpr_t, tpr_t, label="test AUC=%.4f" % test_metrics["roc_auc"])
ax[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("ROC (threshold from val only)")
ax[0].legend(loc="lower right"); ax[0].grid(alpha=0.3)

cm = confusion_matrix(y_test, (y_test_proba >= THRESHOLD).astype(int), labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1])
ax[1].set_xlabel("predicted"); ax[1].set_ylabel("true"); ax[1].set_title("TEST confusion matrix")

plt.tight_layout()
FIG = os.path.join(DRIVE_DIR, "convnext_metrics.png")
plt.savefig(FIG, dpi=130)
print("figure ->", FIG)

In [ ]:
# ---------------------------------------------------------------- artifacts -> Drive
best_model.save(BEST_H5)   # .h5 copy, loadable with the existing verify_models.py tooling

metrics_payload = {
    "model": "ConvNeXt-Tiny",
    "tensorflow_version": tf.__version__,
    "gpu": GPU_NAME,
    "input_shape": [INPUT_SIZE[0], INPUT_SIZE[1], 3],
    "input_range": "[0, 255] float32; Rescaling(1/127.5, -1.0) -> [-1,1] is INSIDE the model",
    "output": "Dense(1, sigmoid); p(high) = Cancer (dx in mel, bcc, akiec)",
    "label_source": "HAM10000_metadata.csv dx column (filenames never used as labels)",
    "split": {"grouping": meta["split_by"].iloc[0], "ratios": "70/15/15", "seed": SEED,
              "train_images": len(df_train), "val_images": len(df_val), "test_images": len(df_test),
              "train_group_overlap": 0},
    "class_counts": {k: v["label_name"].value_counts().to_dict()
                     for k, v in (("train", df_train), ("val", df_val), ("test", df_test))},
    "class_balancing": "tf.data sample_from_datasets, equal weights -> 50/50 stream on TRAIN only",
    "stage1": {"epochs": STAGE1_EPOCHS, "lr": LR_HEAD, "backbone": "frozen"},
    "stage2": {"epochs": STAGE2_EPOCHS, "lr_head": LR_HEAD, "lr_backbone": LR_BACKBONE,
               "unfrozen_stages": to_unfreeze, "label_smoothing": LABEL_SMOOTH},
    "threshold": {
        "selected_on": "VALIDATION ONLY (argmax F1)",
        "value": THRESHOLD,
        "accuracy_optimal_alternative": THRESHOLD_ACC_OPT,
        "sensitivity_ge_0.95_alternative": THRESHOLD_SENS95,
        "app_py_current_threshold": 0.5,
        "app_py_changed": False,
    },
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "test_melanoma_sensitivity": (None if math.isnan(MEL_SENSITIVITY) else MEL_SENSITIVITY),
    "test_classification_report": classification_report(
        y_test, (y_test_proba >= THRESHOLD).astype(int), target_names=CLASS_NAMES,
        output_dict=True, zero_division=0),
    "test_per_dx_detection_rate": per_dx(df_test_probs, THRESHOLD).to_dict(orient="index"),
    "production_model_untouched": True,
    "auto_replaced_production": False,
}

def _json_safe(o):
    if isinstance(o, (np.integer, np.floating, np.bool_)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    return str(o)


with open(os.path.join(DRIVE_DIR, "metrics_convnext.json"), "w") as fh:
    json.dump(metrics_payload, fh, indent=2, default=_json_safe, allow_nan=False)
with open(os.path.join(DRIVE_DIR, "threshold_convnext.json"), "w") as fh:
    json.dump({"threshold": THRESHOLD, "rule": "argmax F1 on validation split only",
               "app_py_remains": 0.5}, fh, indent=2)

for p in (os.path.join(DRIVE_DIR, "metrics_convnext.json"),
          os.path.join(DRIVE_DIR, "threshold_convnext.json")):
    json.load(open(p))     # strict re-parse: fails loudly if anything is not serialisable
print("metrics_convnext.json re-parsed as strict JSON: OK")

print("\nsaved to Drive:")
for p in sorted(glob.glob(os.path.join(DRIVE_DIR, "*"))):
    print("   %-46s %8.2f MB" % (os.path.basename(p),
                                 os.path.getsize(p) / 1e6 if os.path.isfile(p) else 0.0))

In [ ]:
# ---------------------------------------------------------------- verify the saved model loads
verify_paths = [BEST_PATH, BEST_H5]
x_probe = x_chk[:8]
p_ref = best_model.predict(x_probe, verbose=0).reshape(-1)

ok_all = True
for path in verify_paths:
    m = tf.keras.models.load_model(path, compile=False)          # plain load, no custom_objects
    p = m.predict(x_probe, verbose=0).reshape(-1)
    checks = {
        "input_shape_224": tuple(m.input_shape[1:3]) == (224, 224),
        "output_single_unit": tuple(m.output_shape[1:]) == (1,),
        "prob_range_0_1": bool(np.all((p >= 0) & (p <= 1))),
        "matches_training_model": bool(np.allclose(p, p_ref, atol=1e-3)),
        "gives_both_classes": len(set(np.round(p_ref, 3))) > 1,
    }
    print("\n%s  (%.1f MB)" % (os.path.basename(path), os.path.getsize(path) / 1e6))
    for k, v in checks.items():
        print("   %-24s %s" % (k, "PASS" if v else "FAIL"))
    ok_all &= all(checks.values())
    del m

print("\nVERIFIED:", ok_all)
assert ok_all, "saved ConvNeXt model failed the load verification"
assert not AUTOREPLACE_PRODUCTION, "autoreplace guard tripped"
print("Production model skin_cancer_mobilenetv2.h5 was NOT modified.")

---
### 9. Handover — printed only, nothing is changed automatically

The notebook deliberately stops here. It does **not** touch `app.py`, does **not** touch
`model/skin_cancer_mobilenetv2.h5`, and does **not** change `THRESHOLD = 0.5`.

Only after a human has reviewed the metrics should the two lines below be applied
manually.

In [ ]:
# ---------------------------------------------------------------- handover text (no auto-change)
handover = '''DermaAI - ConvNeXt-Tiny handover
==================================
Trained artefact (Google Drive):
  {best}
  {h5}
  size {size:.1f} MB

Frozen decision threshold (chosen on VALIDATION only):
  argmax F1 threshold : {thr:.4f}
  current app.py     : 0.5   <-- unchanged

Metrics at threshold {thr:.4f}
  VALIDATION : acc {va:.4f} | prec {vp:.4f} | recall {vr:.4f} | f1 {vf:.4f} | auc {vauc:.4f}
  TEST       : acc {ta:.4f} | prec {tp_:.4f} | recall {tr:.4f} | f1 {tf:.4f} | auc {tauc:.4f}
  TEST confusion matrix : tn={tn} fp={fp} fn={fn} tp={tp}
  cancer sensitivity    : {sens:.4f}
  melanoma sensitivity  : {mel:.4f}

Manual steps to switch production (ONLY after review) -- app.py:
  1. MODEL_PATH = ... "model", "convnext_tiny_best.h5"
  2. preprocess(): the ConvNeXt file embeds its own Rescaling(1/127.5, -1.0),
     so feed raw RGB in [0, 255] and DROP the "/ 255.0" line.
     The MobileNetV2 model keeps "/ 255.0" -- the two do not share a contract.
  3. Leave THRESHOLD = 0.5 unless you deliberately adopt {thr:.4f}.
  4. tools/verify_models.py can be pointed at the new file; grad_cam() in app.py
     still works because ConvNeXt contains Conv2D layers.

NOT done automatically, on purpose:
  * model/skin_cancer_mobilenetv2.h5 left byte-identical
  * app.py left unchanged
  * no filename-based labels anywhere
'''.format(best=BEST_PATH, h5=BEST_H5, size=os.path.getsize(BEST_H5) / 1e6, thr=THRESHOLD,
           va=val_metrics["accuracy"], vp=val_metrics["precision"], vr=val_metrics["recall"],
           vf=val_metrics["f1"], vauc=val_metrics["roc_auc"],
           ta=test_metrics["accuracy"], tp_=test_metrics["precision"], tr=test_metrics["recall"],
           tf=test_metrics["f1"], tauc=test_metrics["roc_auc"],
           tn=test_metrics["tn"], fp=test_metrics["fp"], fn=test_metrics["fn"], tp=test_metrics["tp"],
           sens=test_metrics["cancer_sensitivity"], mel=MEL_SENSITIVITY)

HANDOVER_PATH = os.path.join(DRIVE_DIR, "handover.txt")
with open(HANDOVER_PATH, "w", encoding="utf-8") as fh:
    fh.write(handover)
print(handover)
print("written ->", HANDOVER_PATH)

# sanity: the production artefacts on the Colab side were never touched
assert AUTOREPLACE_PRODUCTION is False
print("\nNo production file was written by this notebook.")